# Integración de los mercados mayoristas de alimentos en el Perú

**Cuaderno de resultados.** Todo el cálculo está en `scripts/pipeline` y `scripts/analisis`.
Este cuaderno muestra el flujo de EDA y modelado: datos, matriz de diseño, predicciones y
evaluación. Las celdas se ejecutan para no conservar salidas incrustadas de una versión anterior.

| Elemento | Valor |
|---|---|
| Fuente | SISAP–MIDAGRI, módulo Precios de ciudades |
| Unidad analítica | serie región-categoría por mes (27 regiones × 9 categorías) |
| Muestra | 49 041 observaciones limpias |
| Cobertura | 1997–noviembre 2026; estimación ECM 2012–2023 |
| Nowcast | evaluación de desarrollo 2018–2020; holdout enero 2021–octubre 2026 |
| Pregunta | ¿cuánto y con qué rezago responde el precio regional al de Lima? |

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

RAIZ = Path.cwd()
if not (RAIZ / "data").exists():
    RAIZ = RAIZ.parent
TAB = RAIZ / "outputs" / "informe" / "tablas"
AN = RAIZ / "outputs" / "analisis"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

panel = pd.read_csv(RAIZ / "data" / "processed" / "panel_precios_limpio.csv", parse_dates=["fecha"])
print(f"{len(panel):,} filas | {panel.region.nunique()} regiones | {panel.categoria.nunique()} productos")
print(f"{panel.fecha.min():%Y-%m} a {panel.fecha.max():%Y-%m}")

49,041 filas | 27 regiones | 9 productos
1997-01 a 2026-11


## 0. Flujo metodológico

El estudio sigue las cinco fases KDD. La selección y el preprocesamiento alimentan la
transformación; la minería combina el ECM con un torneo de cuatro modelos; la evaluación usa
HAC, bootstrap y Diebold–Mariano.

![](../outputs/informe/figuras/01_flujo_kdd.png)

## 1. Datos y auditoría

La fuente no es un panel balanceado: 1997–2001 solo cubre Lima y la cobertura regional se vuelve
densa en 2012. La tabla del dataset y la auditoría documentan la muestra efectiva.

In [2]:
display(pd.read_csv(TAB / "tabla_01_variables.csv"))
display(pd.read_csv(TAB / "tabla_01b_ejemplo.csv"))

,variable,tipo,unidad,rol,fuente
0,fecha,fecha,mes,índice temporal,SISAP
1,anio,entero,año,derivada,fecha
2,mes,entero,1-12,derivada,fecha
3,region,texto,27 regiones,unidad transversal,SISAP
4,categoria,texto,9 categorías,unidad de producto armonizado,SISAP
5,precio_kg,decimal,S/ por kg,variable de análisis,SISAP normalizado
6,n_registros,entero,conteo,control de agregación,derivada


,fecha,anio,mes,region,categoria,n_registros,precio_kg
0,2008-09-01,2008,9,Amazonas,Aceite,1,5.56
1,2008-10-01,2008,10,Amazonas,Aceite,1,6.39
2,2008-11-01,2008,11,Amazonas,Aceite,1,6.72
3,2008-12-01,2008,12,Amazonas,Aceite,1,6.64
4,2009-01-01,2009,1,Amazonas,Aceite,1,6.34
5,2009-02-01,2009,2,Amazonas,Aceite,1,5.71
6,2009-03-01,2009,3,Amazonas,Aceite,1,5.54
7,2009-04-01,2009,4,Amazonas,Aceite,1,5.66


### Cobertura del panel armonizado

La figura usa las 9 categorías del panel analítico, no el conteo bruto de variedades de la fuente.
Por eso en 2000 muestra 24 regiones y 7 categorías, en concordancia con la Tabla 2.

![](../outputs/informe/figuras/02_cobertura.png)

In [3]:
display(pd.read_csv(TAB / "tabla_02_auditoria.csv"))
display(pd.read_csv(TAB / "tabla_22_verificacion_limpieza.csv"))

,metrica,valor
0,filas,49113
1,regiones,27
2,productos,9
3,nulos,0
4,duplicados fecha-region-producto,0
5,atipicos detectados,72
6,atipicos (%),0.147
7,primer mes,1997-01
8,ultimo mes,2026-11
9,cobertura 2012-2023 (%),85.7


,metrica,valor
0,filas antes,49113.0
1,filas despues,49041.0
2,filas descartadas,72.0
3,atipicos marcados,72.0
4,nulos antes,0.0
5,nulos despues,0.0
6,duplicados antes,0.0
7,duplicados despues,0.0
8,cobertura 2012-2023 antes (%),85.7
9,cobertura 2012-2023 despues (%),85.5


![](../outputs/informe/figuras/23_verificacion_limpieza.png)

## 2. Descripción de los datos

Los precios son asimétricos a la derecha. El log reduce la asimetría en 9 de 9 productos, aunque
la normalidad sigue rechazándose por curtosis. Por eso se trabaja en logaritmos y se evalúa con
MAE y R², no con promedios.

In [4]:
display(pd.read_csv(TAB / "tabla_03_descriptivos.csv").round(3))
display(pd.read_csv(TAB / "tabla_20_transformacion_log.csv").round(3))

,producto,n,media,desv,min,p25,mediana,p75,max,asimetria,curtosis
0,Papa,4435,1.210,0.505,0.331,0.820,1.100,1.500,4.670,1.098,1.753
1,Cebolla,6751,1.370,0.725,0.140,0.850,1.230,1.710,6.670,1.743,5.827
2,Zanahoria,6209,1.298,0.740,0.220,0.810,1.124,1.577,6.930,1.963,5.904
3,Azucar,5388,2.440,0.676,0.960,1.924,2.355,2.800,4.369,0.524,-0.402
4,Arroz,5195,2.711,0.577,1.209,2.326,2.630,3.082,5.400,0.384,-0.006
5,Limon,4920,2.289,1.591,0.270,1.318,2.000,2.750,26.350,3.423,22.498
6,Tomate,4833,1.781,0.780,0.430,1.258,1.608,2.089,8.500,1.631,4.509
7,Aceite,4405,7.309,1.790,3.458,6.361,6.750,7.350,13.654,1.552,1.563
8,Huevos,6905,4.763,1.584,1.800,3.690,4.410,5.300,11.190,1.179,1.003


,producto,asimetria_precio,asimetria_log,curtosis_precio,curtosis_log,reduce_asimetria,reduce_curtosis
0,Papa,1.098,0.106,1.753,-0.527,True,True
1,Cebolla,1.743,-0.092,5.827,0.074,True,True
2,Zanahoria,1.963,0.087,5.904,0.171,True,True
3,Azucar,0.524,-0.063,-0.402,-0.445,True,True
4,Arroz,0.384,-0.222,-0.006,0.041,True,False
5,Limon,3.423,0.057,22.498,0.551,True,True
6,Tomate,1.631,0.163,4.509,0.224,True,True
7,Aceite,1.552,1.113,1.563,0.815,True,True
8,Huevos,1.179,0.438,1.003,-0.108,True,True


![](../outputs/informe/figuras/03_campanas_precio.png)

![](../outputs/informe/figuras/05_qq_log.png)

## 3. Estructura de mercados

La correlación entre regiones no es ni cero ni uno: hay integración parcial. Lima se mueve con
Ica (0,85) y Huánuco (0,77), y casi nada con Tumbes (−0,03) o Loreto (−0,01). La distancia explica
parte de ese gradiente; la zona, por sí sola, no separa.

![](../outputs/informe/figuras/06_correlacion_papa.png)

![](../outputs/informe/figuras/07_correlacion_lima.png)

![](../outputs/informe/figuras/08_integracion_distancia.png)

In [5]:
display(pd.read_csv(TAB / "tabla_13_geografia.csv").round(3))
display(pd.read_csv(TAB / "tabla_12_correlacion_lima.csv").head(10).round(3))

,region,r_lima,distancia_km,zona
0,Ica,0.710,265.676,Costa
1,Huanuco,0.698,251.386,Sierra
2,La Libertad,0.663,489.267,Costa
3,Arequipa,0.579,765.665,Costa
4,Lambayeque,0.521,662.548,Costa
5,Pasco,0.464,174.461,Sierra
6,Ayacucho,0.460,329.959,Sierra
7,Ancash,0.457,285.276,Sierra
8,Tacna,0.411,985.030,Costa
9,Jaen,0.396,731.275,Selva


,producto,region,n,correlacion
0,Papa,Ancash,137,0.376
1,Papa,Andahuaylas,135,0.335
2,Papa,Apurimac,123,0.484
3,Papa,Arequipa,140,0.763
4,Papa,Ayacucho,131,0.502
5,Papa,Cajamarca,131,0.307
6,Papa,Chota,90,0.093
7,Papa,Cusco,92,0.245
8,Papa,Huancavelica,65,0.330
9,Papa,Huanuco,136,0.771


## 4. Estacionariedad y selección de rezagos

El orden de integración es mixto: cebolla es I(0) en Puno y Lima; arroz y huevos son I(1);
aceite es mixto en Puno (ADF de la primera diferencia: p = 0,231) e I(1) en Lima; papa y
zanahoria muestran resultados mixtos según las pruebas. Por eso el modelo se interpreta como un
**ECM en niveles** y la cointegración se trata como evidencia complementaria. El BIC elige q = 0
en los seis productos.

In [6]:
display(pd.read_csv(TAB / "tabla_04_estacionariedad.csv").round(4))
display(pd.read_csv(TAB / "tabla_19_estacionariedad_resumen.csv").round(2))
display(pd.read_csv(TAB / "tabla_05_rezagos.csv").round(4))

,producto,serie,adf_c,adf_ct,kpss_c,adf_diff,decision
0,Papa,Puno,0.0005,0.0015,0.1000,0.0000,I(0)
1,Papa,Lima,0.0005,0.0119,0.0404,0.0000,mixto
2,Cebolla,Puno,0.0000,0.0000,0.1000,0.0000,I(0)
3,Cebolla,Lima,0.0000,0.0000,0.1000,0.0000,I(0)
4,Zanahoria,Puno,0.0100,0.0000,0.0100,0.0000,mixto
5,Zanahoria,Lima,0.0020,0.0005,0.0935,0.0000,I(0)
6,Arroz,Puno,0.3871,0.0198,0.0100,0.0000,I(1)
7,Arroz,Lima,0.6431,0.5043,0.0100,0.0000,I(1)
8,Aceite,Puno,0.0983,0.1650,0.0499,0.2314,mixto
9,Aceite,Lima,0.9911,0.9646,0.0100,0.0000,I(1)


,producto,regiones,i0,i1,mixto,adf_mediana
0,Aceite,25,0,23,2,0.78
1,Arroz,25,0,23,2,0.70
2,Cebolla,25,22,1,2,0.00
3,Huevos,25,0,23,2,0.93
4,Papa,22,6,9,7,0.04
5,Zanahoria,26,6,5,15,0.00


,producto,q_aic,q_bic,q_elegido,aic,bic,lb_p,r2
0,Papa,1,0,0,-165.8837,-157.1679,0.1028,0.3427
1,Cebolla,0,0,0,-108.1257,-99.6168,0.2429,0.5180
2,Zanahoria,0,0,0,-85.8145,-77.6059,0.0375,0.1799
3,Arroz,0,0,0,-400.0392,-391.5304,0.0005,0.1246
4,Aceite,0,0,0,-480.7158,-472.1132,0.0119,0.1592
5,Huevos,0,0,0,-353.5173,-344.7139,0.8180,0.1998


![](../outputs/informe/figuras/21_adf_region.png)

![](../outputs/informe/figuras/22_aic_bic.png)

## 5. Transmisión de precios

Con q = 0, errores HAC y vida media con IC bootstrap:

| Producto | β₀ | γ | Vida media | IC 95 % | R² |
|---|---:|---:|---:|---|---:|
| Papa | 0,307 | −0,306 | 1,90 | 1,35–2,80 | 0,343 |
| Cebolla | 0,509 | −0,277 | 2,14 | 1,55–3,53 | 0,518 |
| Zanahoria | 0,177 | −0,219 | 2,81 | 1,74–6,11 | 0,180 |
| Arroz | 0,123 | −0,164 | 3,87 | 1,42–34,22 | 0,125 |
| Aceite | 0,237 | −0,100 | 6,61 | 4,14–16,35 | 0,159 |
| Huevos | 0,144 | −0,179 | 3,51 | 2,09–9,50 | 0,200 |

El traspaso es incompleto y heterogéneo: la cebolla se ajusta en poco más de dos meses y el
aceite en unos seis meses y medio. El arroz tiene residuos autocorrelacionados y heterocedasticidad;
se reporta con cautela.

In [7]:
display(pd.read_csv(TAB / "tabla_06_ecm_puno.csv").round(3))
display(pd.read_csv(TAB / "tabla_07_ecm_producto.csv").round(3))
display(pd.read_csv(TAB / "tabla_18_diagnostico.csv").round(3))
display(pd.read_csv(TAB / "tabla_11_asimetria.csv").round(3))

,producto,q,n,beta0,beta0_p,gamma,gamma_p,vida_media,vida_ic_inf,vida_ic_sup,r2,p_coint
0,Papa,0,135,0.307,0.000,-0.306,0.000,1.897,1.350,2.795,0.343,0.000
1,Cebolla,0,126,0.509,0.000,-0.277,0.000,2.138,1.552,3.526,0.518,0.000
2,Zanahoria,0,114,0.177,0.017,-0.219,0.000,2.809,1.745,6.108,0.180,0.001
3,Arroz,0,126,0.123,0.171,-0.164,0.045,3.868,1.425,34.222,0.125,0.156
4,Aceite,0,130,0.237,0.013,-0.100,0.000,6.609,4.139,16.348,0.159,0.234
5,Huevos,0,139,0.144,0.018,-0.179,0.000,3.507,2.087,9.501,0.200,0.006


,producto,pares,n_mediana,beta0_mediana,gamma_mediana,vida_mediana,r2_mediana,pares_significativos
0,Cebolla,25,138.0,0.509,-0.299,1.947,0.505,24
1,Papa,22,135.0,0.335,-0.262,2.283,0.324,22
2,Zanahoria,26,126.5,0.227,-0.219,2.807,0.232,26
3,Huevos,25,140.0,0.300,-0.219,2.808,0.286,25
4,Aceite,24,126.5,0.369,-0.168,3.773,0.195,13
5,Arroz,25,137.0,0.172,-0.082,8.100,0.125,18


,producto,n,ljung_box_4,ljung_box_8,breusch_pagan,jarque_bera,se_gamma_ols,se_gamma_hac,r2
0,Papa,135,0.103,0.107,0.354,0.000,0.046,0.059,0.343
1,Cebolla,126,0.243,0.469,0.228,0.359,0.048,0.043,0.518
2,Zanahoria,114,0.038,0.169,0.230,0.334,0.047,0.058,0.180
3,Arroz,126,0.001,0.004,0.000,0.000,0.041,0.082,0.125
4,Aceite,130,0.012,0.002,0.017,0.000,0.025,0.026,0.159
5,Huevos,139,0.818,0.903,0.954,0.000,0.033,0.043,0.200


,producto,n,beta_sube,beta_baja,diferencia,p_igualdad
0,Papa,135,0.489,0.097,0.392,0.097
1,Cebolla,126,0.484,0.531,-0.047,0.756
2,Zanahoria,114,0.354,-0.007,0.361,0.110
3,Arroz,126,0.012,0.285,-0.273,0.522
4,Aceite,130,0.328,-0.010,0.337,0.424
5,Huevos,139,0.237,-0.013,0.250,0.238


![](../outputs/informe/figuras/09_vida_media_puno.png)

![](../outputs/informe/figuras/10_vida_media_mapa.png)

## 6. Torneo de modelos y representación

El objetivo principal es el **nowcast contemporáneo**: estimar el cambio del mes corriente con
información del mismo mes. Random Forest queda primero por rango promedio de R² (**1,50**, frente a
1,75 de Ridge), pero ese ranking de desarrollo se calcula solo entre cuatro productos que cumplen
los mínimos de muestra. Los modelos se entrenan con observaciones anteriores a 2018 y se evalúan
en la ventana calendario 2018–2020; el holdout se extiende de enero de 2021 a octubre de 2026. El
panel contiene noviembre, pero su fecha de cierre/publicación no está verificada y el corte
configurado se mantiene en octubre. En el holdout, los seis productos se evalúan: Random Forest
tiene R² positivo en papa (0,270), cebolla (0,218) y zanahoria (0,080). Para el pronóstico a un mes,
Ridge y OLS mejoran frente a la caminata aleatoria en papa, arroz, cebolla y huevos; OLS mejora
apenas 0,7 % en zanahoria y Ridge empeora; ambos empeoran en aceite. La tabla de pronóstico del proyecto compara los cuatro modelos.

In [8]:
display(pd.read_csv(TAB / "tabla_08_torneo.csv").round(3))
display(Markdown((AN / "06_modelos" / "torneo_ganador.md").read_text(encoding="utf-8")))
holdout = pd.read_csv(TAB / "tabla_09_ganador.csv")
display(holdout[(holdout.periodo == "holdout") & (holdout.modelo == "Random Forest")][["producto", "r2", "acierto", "n"]].round(3))

,modelo,rank_promedio,r2_medio,r2_mediano,acierto_medio,productos_rankeados
0,Random Forest,1.50,-0.687,-0.097,46.879,4
1,Ridge,1.75,-0.661,-0.137,46.228,4
2,OLS,2.75,-0.689,-0.153,46.228,4
3,LightGBM,4.00,-2.434,-0.544,42.785,4


# Resultado del ranking de nowcast

**Objetivo principal:** anticipacion contemporanea del cambio mensual.

- **Modelo adoptado:** Random Forest
- Rango promedio por R2 en desarrollo: 1.50
- R2 mediano en desarrollo: -0.097
- Ranking calculado sobre 4 productos con muestras suficientes: Papa, Arroz, Aceite, Huevos
- Excluidos del ranking por no alcanzar los mínimos muestrales: Cebolla, Zanahoria
- Mínimos por producto para rankear: 60 observaciones de entrenamiento y 20 de evaluación
- Holdout: 3 de 6 productos con R2 > 0; 3 con acierto > 55 %
- Compuerta: superada

### Holdout por producto

| Producto | R2 | Acierto | n |
|---|---:|---:|---:|
| Papa | 0.270 | 69.5 % | 60 |
| Cebolla | 0.218 | 64.4 % | 60 |
| Zanahoria | 0.080 | 52.9 % | 52 |
| Arroz | -0.842 | 41.9 % | 31 |
| Aceite | -0.103 | 50.0 % | 24 |
| Huevos | -0.013 | 55.3 % | 38 |

## Justificacion

La ventana de desarrollo es 2018-2020 (36 meses calendario) y se usa para evaluar/rankear; cada modelo se entrena con las observaciones disponibles antes de 2018. Solo se rankean productos con al menos 60 observaciones de entrenamiento y 20 de evaluación. El R2 es la metrica principal del nowcast porque el MAE sobre cambios favorece predecir cero. El holdout 2021-01 a 2026-10 se uso una sola vez; noviembre queda fuera porque no se verifico su fecha de cierre/publicacion.

Por que no gana LightGBM si tiene mayor acierto en el holdout? LightGBM acierta el signo en 62.2 % de los meses, frente a 58.0 % de Random Forest, pero el acierto es una lectura binaria: trata igual un error de 0,1 % que uno de 5 %. El ganador se eligio en desarrollo con R2, donde Random Forest tiene rango promedio 1.50 y acierto medio 46.9 %, contra 4.00 y 42.8 % de LightGBM. En el holdout Random Forest tiene menor MAE que LightGBM en 5 de 6 productos y R2 positivo en papa, cebolla y zanahoria; LightGBM tiene R2 muy negativo en arroz y zanahoria. Cambiar el ganador segun el holdout seria seleccionar sobre el conjunto de prueba. La matriz de confusion se reporta como diagnostico complementario, no como criterio de seleccion.

## Pronostico a un mes: holdout

Cambio porcentual del MAE frente a la caminata aleatoria: positivo = mejora; negativo = empeora.

| Producto | N | OLS | Ridge | Random Forest | LightGBM |
|---|---:|---:|---:|---:|---:|
| Papa | 62 | +14.0 % | +12.2 % | -65.7 % | -58.8 % |
| Cebolla | 62 | +8.2 % | +9.7 % | -8.4 % | -14.1 % |
| Zanahoria | 53 | +0.7 % | -1.9 % | -65.0 % | -73.0 % |
| Arroz | 36 | +6.3 % | +9.9 % | -282.8 % | -351.0 % |
| Aceite | 24 | -33.9 % | -33.1 % | -303.3 % | -307.6 % |
| Huevos | 38 | +6.7 % | +6.2 % | -148.0 % | -142.4 % |

Ridge y OLS mejoran en papa, arroz, cebolla y huevos; OLS mejora marginalmente en zanahoria (+0,7 %), mientras Ridge empeora (−1,9 %); ambos empeoran en aceite. Random Forest y LightGBM empeoran frente al baseline en los seis productos. La referencia de Lima aporta valor contemporaneo, pero no ofrece un pronostico general a un mes.


,producto,r2,acierto,n
23,Papa,0.270,69.492,60
28,Cebolla,0.218,64.407,60
33,Zanahoria,0.080,52.941,52
38,Arroz,-0.842,41.935,31
43,Aceite,-0.103,50.000,24
48,Huevos,-0.013,55.263,38


### Matriz de diseño y evaluación por dirección

Antes de mirar el ranking conviene ver **qué datos consume el modelo**. Para papa en Puno, cada fila
es un mes y contiene el cambio de Lima (`dl0`), sus rezagos, los rezagos propios y los armónicos.
La columna `y_cambio_log` es el objetivo; `split` indica si la fila cae en entrenamiento, desarrollo
o holdout. Es la matriz que se usa en `modelos.py`, no una tabla resumen inventada después.


In [9]:
diseno = pd.read_csv(TAB / "tabla_24_matriz_diseno.csv")
display(diseno.round(3))
confusion = pd.read_csv(TAB / "tabla_25_confusion_modelos.csv")
display(confusion.round(1))

,fecha,producto,region,dl0,dl1,dl2,dl3,dp1,dp2,sin1,cos1,y_cambio_log,split
0,2012-06-01,Papa,Puno,0.000,-0.134,-0.012,-0.211,0.000,-0.101,0.500,-0.866,-0.019,entrenamiento
1,2012-07-01,Papa,Puno,0.108,0.000,-0.134,-0.012,-0.019,0.000,0.000,-1.000,-0.030,entrenamiento
2,2012-08-01,Papa,Puno,0.000,0.108,0.000,-0.134,-0.030,-0.019,-0.500,-0.866,0.040,entrenamiento
3,2012-09-01,Papa,Puno,-0.066,0.000,0.108,0.000,0.040,-0.030,-0.866,-0.500,-0.081,entrenamiento
4,2026-07-01,Papa,Puno,-0.160,-0.211,-0.168,-0.194,-0.164,-0.025,0.000,-1.000,-0.067,holdout
5,2026-08-01,Papa,Puno,0.419,-0.160,-0.211,-0.168,-0.067,-0.164,-0.500,-0.866,-0.038,holdout
6,2026-09-01,Papa,Puno,0.381,0.419,-0.160,-0.211,-0.038,-0.067,-0.866,-0.500,0.217,holdout
7,2026-10-01,Papa,Puno,-0.009,0.381,0.419,-0.160,0.217,-0.038,-1.000,0.000,0.159,holdout


,modelo,TP,FN,FP,TN,n,acierto
0,LightGBM,97,53,46,66,262,62.2
1,OLS,99,51,56,56,262,59.2
2,Random Forest,92,58,52,60,262,58.0
3,Ridge,99,51,56,56,262,59.2


**Cómo se lee.** `dl0` es el cambio de Lima del mismo mes; `dp1`/`dp2` son cambios propios
rezagados; `sin1`/`cos1` capturan estacionalidad. La tabla muestra cuatro filas del inicio y cuatro
del final; el CSV contiene la matriz completa. La segunda tabla convierte el signo de la predicción
en una decisión “sube/baja”: TP y TN son aciertos; FP y FN son errores de dirección. Los meses con
cambio exactamente cero se excluyen porque no tienen dirección.

![](../outputs/informe/figuras/24_matriz_confusion.png)

**¿Por qué Random Forest sigue siendo el ganador?** LightGBM acierta el signo en 62,2 % de los meses
del holdout, contra 58,0 % de Random Forest, pero el acierto es binario: no distingue un error de
0,1 % de uno de 5 %. El ganador se eligió en desarrollo con R², donde Random Forest tiene rango
promedio 1,50 y acierto medio 46,9 %, frente a 4,00 y 42,8 % de LightGBM. En el holdout Random Forest
tiene menor MAE en cinco de seis productos y R² positivo en papa, cebolla y zanahoria; LightGBM
tiene R² muy negativo en arroz (−4,04) y zanahoria (−0,28). Cambiar el ganador con el holdout sería
seleccionar sobre el conjunto de prueba. La matriz de confusión es un diagnóstico complementario.


![](../outputs/informe/figuras/11_ranking_torneo.png)

![](../outputs/informe/figuras/12_ganador_vs_observado.png)

**Cómo se lee la Figura 15.** Ahora no se muestra solo el ganador: se comparan los cuatro
modelos con el cambio observado en los seis productos del holdout. Una línea que sigue el signo pero
suaviza los picos tiene buen acierto direccional y peor MAE; una que se pega a los picos puede
sobreajustar. Esa tensión es la que resumen la tabla de métricas y la matriz de confusión.

Las curvas de aprendizaje son de cebolla. Los tamaños 33, 46, 56 y 67 son observaciones
mensuales de entrenamiento anteriores a 2018, no la duración de la ventana de desarrollo. La
ventana calendario de evaluación es 2018–2020; para cebolla hay 17 observaciones válidas en ese
tramo. Random Forest reduce su MAE y aumenta su R² al crecer el entrenamiento; LightGBM también
mejora y supera a OLS/Ridge en el último tramo. Es una curva de un producto, no una regla para todos
los mercados.

![](../outputs/informe/figuras/13_curvas_aprendizaje.png)

### Representación del Random Forest

El primer árbol del bosque se parte por el cambio de Lima (`dl0`), luego por el cambio propio
(`dp2`) y por la estacionalidad (`sin1`). Es la misma señal que captura el ECM.

![](../outputs/informe/figuras/15_arbol_rf.png)

![](../outputs/informe/figuras/17_importancia_rf.png)

![](../outputs/informe/figuras/18_pdp.png)

### Representación de LightGBM

El árbol sustituto aproxima el boosting: un árbol simple entrenado sobre las predicciones del
ensemble. Muestra qué variables domina el modelo, sin necesidad de visualizar cientos de árboles.

![](../outputs/informe/figuras/16_arbol_lgbm.png)

## 7. Robustez

La conclusión no es uniforme en todos los escenarios. Con q2, el N baja de 114 a 112 en zanahoria
por dos meses afectados por un hueco previo en la serie de Lima; en cebolla, arroz, aceite y huevos
se mantiene. El arroz pierde significancia con q2 (p = 0,1234) y al excluir 2021–2023 (p = 0,0822),
pero vuelve a ser significativo al extender la ventana a 2008 (p = 0,0461; N = 142). En aceite, la
ventana 2008 eleva N de 130 a 168; al excluir el shock, la vida media sube de 6,6 a 10,1 meses.
Papa amarilla es una variedad fuente adicional, fuera de las nueve categorías; tiene seis meses de
cruce con Lima y se incluye solo como sensibilidad.

In [10]:
robustez = pd.read_csv(TAB / "tabla_10_robustez.csv")
robustez["escenario"] = robustez["escenario"].replace({
    "sin_shock": "sin shock", "ventana_2008": "ventana 2008",
    "papa_amarilla": "papa amarilla"})
display(robustez.round(3))

,producto,escenario,q,n,beta0,gamma,vida_media,r2,p_gamma
0,Papa,base,0,135,0.307,-0.306,1.897,0.343,0.000
1,Papa,q0,0,135,0.307,-0.306,1.897,0.343,0.000
2,Papa,q1,1,135,0.261,-0.254,2.368,0.362,0.000
3,Papa,q2,2,135,0.265,-0.238,2.552,0.366,0.000
4,Papa,sin shock,0,99,0.295,-0.327,1.747,0.360,0.000
5,Papa,ventana 2008,0,135,0.307,-0.306,1.897,0.343,0.000
6,Cebolla,base,0,126,0.509,-0.277,2.138,0.518,0.000
7,Cebolla,q0,0,126,0.509,-0.277,2.138,0.518,0.000
8,Cebolla,q1,1,126,0.497,-0.263,2.268,0.519,0.000
9,Cebolla,q2,2,126,0.497,-0.267,2.236,0.519,0.000


![](../outputs/informe/figuras/14_robustez.png)

## 8. Veredicto

- **Sí** hay integración parcial y transmisión. En la especificación base con q elegido por BIC, γ
  es negativo y significativo en los seis productos de Puno; el arroz pierde significancia con q2
  (p = 0,1234) y sin el shock (p = 0,0822).
- **Sí** hay valor predictivo contemporáneo específico. Random Forest queda primero en el ranking
  de desarrollo, que se calcula sobre cuatro productos elegibles; el holdout incluye los seis y
  muestra R² positivo en papa, cebolla y zanahoria.
- **El pronóstico a un mes es heterogéneo:** Ridge y OLS mejoran frente a la caminata aleatoria en
  papa, arroz, cebolla y huevos; OLS mejora apenas 0,7 % en zanahoria y Ridge empeora; ambos
  empeoran en aceite. Random Forest y LightGBM empeoran frente a la caminata aleatoria en los seis
  productos (Tabla 17).
- La geografía se asocia con integración, especialmente por distancia; la zona por sí sola no basta.

El detalle metodológico está en `docs/metodologia.md`; el plan y las decisiones, en
`docs/plan_desarrollo.md`.